# Emerging Technologies Problems Notebook

In [4]:
import random
from itertools import product
from qiskit import QuantumCircuit

## Problem 1: Generating Random Boolean Functions

**Brief:** The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

**random_constant_balanced() function:**

The random_constant_balanced function generates and returns a random Boolean function of four inputs (a, b, c, d) that satisfies the Deutsch–Jozsa promise: the function is guaranteed to be either constant or balanced. It first randomly decides whether to create a constant or balanced function. If constant, it randomly selects True or False and returns a function that ignores its inputs and always outputs that value. If balanced, it generates all 16 possible input combinations for four Boolean variables and randomly selects exactly 8 of them to return True, with the remaining 8 returning False. The returned function then checks whether its input tuple belongs to the selected set.

In [5]:
def random_constant_balanced():
    # Decide whether function is constant or balanced
    is_constant = random.choice([True, False])
    
    if is_constant:
        # Create a constant function
        value = random.choice([True, False])
        
        def constant_function(a, b, c, d):
            return value
        
        return constant_function
    
    else:        
        # Create a balanced function
        # Generate all 16 input combinations
        inputs = list(product([False, True], repeat=4))
        
        # Randomly choose 8 combinations to return True
        true_inputs = set(random.sample(inputs, 8))
        
        # check whether arguments in the list are in the selected 'True' set
        def balanced_function(a, b, c, d):
            return (a, b, c, d) in true_inputs
        
        return balanced_function

In [6]:
f = random_constant_balanced()

print(f(False, False, False, False))
print(f(True, False, True, True))

False
False


## Problem 2: Classical Testing for Function Type

**Brief:** Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

### determine_constant_balanced(f) Function: 

The function determine_constant_balanced determines whether a given 4-input Boolean function f is constant or balanced by evaluating the function on every possible input combination. Using product([False, True], repeat=4), the code generates all 16 possible combinations of four Boolean inputs. It then calls f on each combination and counts how many times the function returns True. 

If the count is either 0 or 16, all outputs are the same and the function is constant; otherwise, since the problem guarantees the function is either constant or balanced, it must be balanced (8 True and 8 False). The algorithm runs in n-input Boolean function because it checks every possible input. In the worst case, the function must call f 16 times to be completely certain of the result, since even after 15 identical outputs the final input could still determine whether the function is constant or balanced. This classical cost highlights the motivation behind Deutsch's algorithm, which can solve the problem with only one evaluation of the function on a quantum computer.

In [7]:
from itertools import product 

def determine_constant_balanced(f):
    """
    Determines whether a given 4-input Boolean function f
    is constant or balanced.
    """

    # Generate all 16 possible input combinations for four Boolean variables
    inputs = list(product([False, True], repeat=4))

    # Counter to track how many times f returns True.
    true_count = 0

    # Evaluate f on every possible input
    for inp in inputs:
        if f(*inp):
            true_count += 1
            
    # If all outputs are the same → constant
    if true_count == 0 or true_count == 16:
        return "constant"
    else:
        # Since the problem guarantees constant or balanced,
        # the only remaining possibility is balanced (8 True, 8 False)
        return "balanced"

## Problem 2 Testing

In [8]:
# ---- Test 1: Constant False ----
def constant_false(a, b, c, d):
    return False

assert determine_constant_balanced(constant_false) == "constant"
print("Test 1 passed: constant_false correctly identified.")


# ---- Test 2: Constant True ----
def constant_true(a, b, c, d):
    return True

assert determine_constant_balanced(constant_true) == "constant"
print("Test 2 passed: constant_true correctly identified.")


# ---- Test 3: Balanced Function (Parity Example) ----
# Returns True when an even number of inputs are True
def balanced_example(a, b, c, d):
    return (a + b + c + d) % 2 == 0

# This is balanced: exactly 8 inputs return True
assert determine_constant_balanced(balanced_example) == "balanced"
print("Test 3 passed: balanced_example correctly identified.")


# ---- Test 4: Randomly Generated Functions ----
# Tests compatibility with Problem 1 generator
for i in range(5):
    f = random_constant_balanced()
    result = determine_constant_balanced(f)
    print(f"Random test {i+1}: Function classified as {result}")

print("Random tests executed successfully.")

Test 1 passed: constant_false correctly identified.
Test 2 passed: constant_true correctly identified.
Test 3 passed: balanced_example correctly identified.
Random test 1: Function classified as balanced
Random test 2: Function classified as constant
Random test 3: Function classified as balanced
Random test 4: Function classified as constant
Random test 5: Function classified as balanced
Random tests executed successfully.


## Problem 3: Quantum Oracles

**Brief:** Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

### f(x) = 0 constant function: 


f(x) = 0: The circuit does nothing because the output qubit should not change.

In [9]:
# f(x) = 0  (constant function)
def oracle_f0():
    qc = QuantumCircuit(2)
    # Does nothing because f(x) = 0 → y ⊕ 0 = y
    return qc

### f1(x) = 1  constant one function:

In [10]:
def oracle_f1(qc: QuantumCircuit, x: int, y: int) -> None:
    
    qc.x(y)

### f2(x) = x  - identity / balanced function:

In [11]:
def oracle_f2(qc: QuantumCircuit, x: int, y: int) -> None:

    qc.cx(x, y)

### f3(x) = NOT x  - negation / balanced function:

In [12]:
def oracle_f3(qc: QuantumCircuit, x: int, y: int) -> None:
    
    qc.x(x)
    qc.cx(x, y)
    qc.x(x)

### Testing Circuit Functions

In [13]:
# Demonstrate the circuit
print("Oracle for f(x) = 0")
print(oracle_f0().draw())

Oracle for f(x) = 0
     
q_0: 
     
q_1: 
     


## Problem 4: Deutsch's Algorithm with Qiskit

**Brief:** Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm

**Brief:** The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.